# 📈 Anti-Hype Engine: 교차 검증 기반 기업 가치 기만 탐지 시스템

이 노트북에서는 금융 뉴스 감성 지수와 주가지수를 바탕으로 측정된 **시장 과열도(Market Hype)**와 DART 사업보고서 공시를 통해 추출한 **기업의 실체적 역량(Substance)**을 교차 검증하여, 부풀려진 기업 가치(Hype)와 저평가 우량주(Undervalued)를 탐지하는 **Anti-Hype Engine** 파이프라인을 구축합니다.

### 🔍 분석 모델 및 데이터 연동 정보
- **뉴스 감성 분석**: `snunlp/KR-FinBert-SC` (금융 특화 한국어 감성 분류 모델)
- **시장 가격 데이터**: `yfinance` (실제 야후 파이낸스 일별 주가 연동)
- **실체적 역량 지표 (DART 기준)**:
  1. **RD_Ratio_Pct**: 매출액 대비 R&D 투자 비율 (%)
  2. **RD_Staff**: R&D 전담 연구원 수 (명)
  3. **PhD_Count**: 박사급 핵심 연구원 수 (명)
  4. **Unconfirmed_Disclosures**: 3개년 누적 미확정 반복 공시 횟수 (기만 리스크 요소)

## 🛠️ 1. 필수 라이브러리 설치 및 불러오기

Hugging Face의 `transformers`, 딥러닝 백엔드 `torch`, 실제 주가 수집을 위한 `yfinance`, 시각화를 위한 `matplotlib/seaborn` 등을 불러옵니다.

In [ ]:
# 필요한 라이브러리가 설치되어 있지 않다면 아래 주석을 해제하고 실행하세요.
# !pip install transformers torch pandas numpy matplotlib seaborn yfinance

In [ ]:
import os
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# 한글 깨짐 방지를 위한 matplotlib 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

## 🤖 2. KR-FinBERT 금융 감성 분석 모델 로드

In [ ]:
model_name = "snunlp/KR-FinBert-SC"

print("금융 특화 BERT 모델 및 토크나이저를 불러오는 중...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
print("✅ 모델 로드 완료!")

## 📊 3. DART 기반 실체 역량(Substance) 데이터 정의

공시 데이터 기준 핵심 제약/바이오 기업(삼천당제약, 유한양행, 삼성바이오로직스, 한미약품 및 가상의 저평가 바이오벤처)의 R&D 지표들을 정의합니다.

In [ ]:
companies_data = {
    "Ticker": ["000250.KQ", "000100.KS", "207940.KS", "128940.KS", "BENCH.QA"],
    "Name": ["삼천당제약", "유한양행", "삼성바이오로직스", "한미약품", "저평가바이오벤처"],
    "RD_Ratio_Pct": [6.8, 11.5, 7.5, 13.2, 25.0],        # 매출액 대비 R&D 비율 (%)
    "RD_Staff": [35, 250, 500, 280, 80],                  # R&D 전담 연구원 수 (명)
    "PhD_Count": [1, 80, 150, 75, 35],                    # 박사급 핵심 연구원 수 (명)
    "Unconfirmed_Disclosures": [23, 0, 0, 1, 1]           # 3개년 미확정 공시 횟수
}

df_substance = pd.DataFrame(companies_data)
df_substance

## 📈 4. yfinance 연동을 통한 실제 주가 데이터 수익률 계산

삼천당제약 사태의 폭등과 폭락이 가장 극적이었던 기간인 **2025년 1월부터 2026년 4월까지**의 실제 주가 데이터를 다운로드하여 누적 주가 변화율을 측정합니다.

In [ ]:
start_date = "2025-01-01"
end_date = "2026-04-30"

returns = {}
for idx, row in df_substance.iterrows():
    ticker = row["Ticker"]
    name = row["Name"]
    if ticker == "BENCH.QA":
        # 가상의 벤처기업은 주가 움직임이 잔잔했던 상태로 가정 (시장 무관심/저평가)
        returns[name] = 0.05
        continue
    try:
        # yfinance 다운로드
        stock_df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)
        if not stock_df.empty:
            if isinstance(stock_df.columns, pd.MultiIndex):
                stock_df.columns = stock_df.columns.droplevel(1)
            close_prices = stock_df["Close"]
            # 기간 누적 수익률 계산
            cum_return = (close_prices.iloc[-1] - close_prices.iloc[0]) / close_prices.iloc[0]
            returns[name] = float(cum_return)
        else:
            returns[name] = 0.0
    except Exception as e:
        print(f"{name} 주가 다운로드 실패: {e}")
        returns[name] = 0.0

df_substance["Stock_Return"] = df_substance["Name"].map(returns)
df_substance

## 📰 5. KR-FinBERT 기반 뉴스 감성 분석 점수 산출

각 기업의 대표 보도자료 및 뉴스 헤드라인 샘플을 정의하고, KR-FinBERT를 활용하여 긍정/부정 스코어를 산출합니다.

In [ ]:
news_by_company = {
    "삼천당제약": [
        "삼천당제약, 세계 최초 경구용 세마글루타이드 상업화 임박 기대감 최고조",
        "삼천당제약, 다이이찌산쿄와 초대형 파트너십 체결 공시... 글로벌 영토 확장",
        "삼천당제약, 먹는 인슐린 혁신 신약 게임체인저 등극 전망",
        "삼천당제약, 독자 플랫폼 기술 수출 논의 본격화... 주가 급등세",
        "삼천당제약, 아일리아 바이오시밀러 미국 출시 지연 루머 부인... 주가 방어 총력"
    ],
    "유한양행": [
        "유한양행, 신약 연구개발 순항 중... 올해 실적 안정적 우상향 전망",
        "유한양행, 폐암 신약 렉라자 미국 FDA 승인 후 시장 안착 순조로워",
        "유한양행, 분기 영업이익 소폭 감소했으나 R&D 투자는 지속 확대"
    ],
    "삼성바이오로직스": [
        "삼성바이오로직스, 4공장 본격 가동에 따른 수주 잔고 사상 최대치 갱신",
        "삼성바이오로직스, 대형 제약사들과 위탁생산 계약 추가 체결... 안정적 성장세",
        "삼성바이오로직스, 글로벌 빅파마 파트너십 강화 및 바이오시밀러 시장 주도"
    ],
    "한미약품": [
        "한미약품, 차세대 비만 치료제 신약 임상 진행... 시장 선점 기대",
        "한미약품, 신약 파이프라인 반환 리스크 극복하고 해외 라이선스 성과 가시화",
        "한미약품, R&D 투자 확대 기조 유지... 탄탄한 실적 기반 지탱"
    ],
    "저평가바이오벤처": [
        "신약 벤처 A사, 획기적인 R&D 특허 확보에도 자금 조달 우려에 시장 소외",
        "바이오 벤처 A사, 묵묵한 학회 발표에도 거래량 부족으로 주가 약세 지속"
    ]
}

sentiments = {}
for name, headlines in news_by_company.items():
    scores = []
    for text in headlines:
        pred = classifier(text)[0]
        label = pred['label'].lower()
        score = pred['score']
        # 긍정: +score, 부정: -score, 중립: 0
        if label == 'positive':
            val = score
        elif label == 'negative':
            val = -score
        else:
            val = 0.0
        scores.append(val)
    sentiments[name] = np.mean(scores) if scores else 0.0

df_substance["News_Sentiment"] = df_substance["Name"].map(sentiments)
df_substance

## 🧮 6. 정규화(Normalization) 및 최종 스코어 산출

지표들을 0~1 사이로 Min-Max 스케일링한 후 가중 합산하여 **Core Substance Score**와 **Market Hype Score**를 산출합니다.

In [ ]:
# Min-Max 정규화 함수 정의
def min_max_scale(series, invert=False):
    s_min = series.min()
    s_max = series.max()
    if s_max == s_min:
        return series * 0.0 + 0.5
    scaled = (series - s_min) / (s_max - s_min)
    if invert:
        scaled = 1.0 - scaled
    return scaled

# 1. 실체 역량(Substance) 스코어 계산
# 미확정 공시는 횟수가 많을 수록 Substance 마이너스 요인이므로 invert=True
sub_rd_ratio = min_max_scale(df_substance["RD_Ratio_Pct"])
sub_rd_staff = min_max_scale(df_substance["RD_Staff"])
sub_phd = min_max_scale(df_substance["PhD_Count"])
sub_discl = min_max_scale(df_substance["Unconfirmed_Disclosures"], invert=True)

df_substance["Substance_Score"] = (
    sub_rd_ratio * 0.3 + 
    sub_rd_staff * 0.2 + 
    sub_phd * 0.3 + 
    sub_discl * 0.2
)

# 2. 시장 과열도(Hype) 스코어 계산
hype_sentiment = min_max_scale(df_substance["News_Sentiment"])
hype_return = min_max_scale(df_substance["Stock_Return"])

df_substance["Hype_Score"] = (
    hype_sentiment * 0.5 + 
    hype_return * 0.5
)

# Hype Index = Hype Score - Substance Score (괴리 폭)
df_substance["Hype_Index"] = df_substance["Hype_Score"] - df_substance["Substance_Score"]

df_substance[["Name", "Substance_Score", "Hype_Score", "Hype_Index"]].sort_values(by="Hype_Index", ascending=False)

## 📊 7. Substance vs Hype 2D 검증 매트릭스 시각화

In [ ]:
plt.figure(figsize=(11, 8.5))

# 사분면 가이드라인 경계값
x_mid, y_mid = 0.5, 0.5

# 사분면 영역 배경 색칠
plt.axvspan(0, x_mid, ymin=y_mid, ymax=1, color='#FFCCCC', alpha=0.3, label='HYPE (부풀려진 기만/경고 영역)')
plt.axvspan(x_mid, 1, ymin=0, ymax=y_mid, color='#CCFFCC', alpha=0.3, label='UNDERVALUED (저평가 우량주 영역)')
plt.axvspan(x_mid, 1, ymin=y_mid, ymax=1, color='#CCE5FF', alpha=0.2, label='FAIR VALUE (고성장/적정 가치)')
plt.axvspan(0, x_mid, ymin=0, ymax=y_mid, color='#F0F0F0', alpha=0.3, label='NEGLECTED (소외/비우량 영역)')

# 기업 분포 산점도 시각화
sns.scatterplot(
    data=df_substance, 
    x="Substance_Score", 
    y="Hype_Score", 
    s=350, 
    hue="Name", 
    palette="Set1", 
    edgecolor="black", 
    linewidth=2.5,
    style="Name",
    markers=['o', 's', '^', 'D', 'P']
)

# 기준선
plt.axvline(x=x_mid, color='gray', linestyle='--', linewidth=1.5)
plt.axhline(y=y_mid, color='gray', linestyle='--', linewidth=1.5)

# 데이터 라벨 추가
for i, row in df_substance.iterrows():
    plt.text(
        row["Substance_Score"] + 0.02, 
        row["Hype_Score"] + 0.01, 
        row["Name"], 
        fontsize=12, 
        weight='bold'
    )

plt.xlim(-0.05, 1.05)
plt.ylim(-0.05, 1.05)
plt.xlabel("실체 역량 점수 (Core Substance Score)", fontsize=13, weight='bold')
plt.ylabel("시장 과열 지수 (Market Hype Score)", fontsize=13, weight='bold')
plt.title("Anti-Hype Engine: 실체 역량 대비 시장 과열도 교차 검증 매트릭스", fontsize=16, weight='bold', pad=20)
plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## 📉 8. 삼천당제약 Hype Gap 시계열 시뮬레이션 및 시각화

삼천당제약의 실제 주가 움직임과 매칭하여, 2025년 말 호재성 언론 플레이와 미확정 공시로 극대화되었던 Hype Gap(시장 버블과 실체 역량 간의 괴리)이 2026년 초 폭락과 불성실 공시법인 지정으로 무너지는 동향을 시계열로 표현합니다.

In [ ]:
ticker = "000250.KQ"
try:
    sct_df = yf.download(ticker, start="2025-07-01", end="2026-04-30", auto_adjust=False)
    if not sct_df.empty:
        if isinstance(sct_df.columns, pd.MultiIndex):
            sct_df.columns = sct_df.columns.droplevel(1)
        
        # 월별 종가 데이터 리샘플링
        sct_monthly = sct_df.resample('ME').last()
        dates = sct_monthly.index
        
        # 월별 시뮬레이션된 뉴스 Hype 감성 점수 (25년말 ~ 26년초 폭등 시 극대화)
        # 2025-07 ~ 2026-04 (10개월)
        simulated_hype_sentiment = [0.4, 0.45, 0.65, 0.85, 0.95, 0.9, 0.8, 0.55, 0.25, 0.15]
        
        # 시각화 기간 일치화
        n_months = len(dates)
        if n_months > len(simulated_hype_sentiment):
            simulated_hype_sentiment = simulated_hype_sentiment + [0.15] * (n_months - len(simulated_hype_sentiment))
        else:
            dates = dates[:len(simulated_hype_sentiment)]
            sct_monthly = sct_monthly.iloc[:len(simulated_hype_sentiment)]
            
        # 주가 정규화 (0~1)
        norm_price = (sct_monthly["Close"] - sct_df["Close"].min()) / (sct_df["Close"].max() - sct_df["Close"].min())
        norm_price = norm_price.values
        
        # Market Hype Score 산출: (주가 정규화 + 뉴스 감성) / 2
        monthly_hype_score = (norm_price + np.array(simulated_hype_sentiment[:len(dates)])) / 2.0
        
        # 삼천당제약의 실체 역량 지표 (기간 내내 1명 박사 등 낮은 역량 고정)
        substance_score = np.array([0.15] * len(dates))
        
        # 시계열 플롯 그리기
        fig, ax1 = plt.subplots(figsize=(13, 6.5))
        
        # 축 1: 실제 주가 종가 변동
        ax1.plot(dates, sct_monthly["Close"], color='black', linewidth=3, label='삼천당제약 실제 주가 (종가)')
        ax1.set_xlabel('날짜 (Date)', fontsize=12, weight='bold')
        ax1.set_ylabel('주가 (원)', color='black', fontsize=12, weight='bold')
        ax1.tick_params(axis='y', labelcolor='black')
        
        # 축 2: 지수 스코어 (이중 축)
        ax2 = ax1.twinx()
        ax2.plot(dates, monthly_hype_score, color='#FF3333', linestyle='--', marker='o', markersize=8, linewidth=2, label='Market Hype (시장 과열 지수)')
        ax2.plot(dates, substance_score, color='#3333FF', linestyle='-.', marker='x', markersize=8, linewidth=2, label='Core Substance (공시 실체 지수)')
        
        # Hype Gap 채우기
        ax2.fill_between(
            dates, substance_score, monthly_hype_score, 
            where=(monthly_hype_score > substance_score), 
            color='red', alpha=0.15, label='Hype Gap (기만/거품 구간)'
        )
        
        ax2.set_ylabel('지수 스코어 (0.0 ~ 1.0)', color='red', fontsize=12, weight='bold')
        ax2.tick_params(axis='y', labelcolor='red')
        ax2.set_ylim(-0.05, 1.05)
        
        # 범례 통합
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)
        
        plt.title("삼천당제약: 시간 경과에 따른 Hype Gap (시장 과열 vs DART 실체) 추이", fontsize=16, weight='bold', pad=20)
        plt.grid(True, linestyle=':', alpha=0.5)
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f"시계열 차트 생성 실패: {e}")